**`requests.Session` — Practical Guide with Real API Examples**

[Docs](https://requests.readthedocs.io/en/latest/api/#requests.Session)

The `requests.Session` class is one of the most important parts of the Python Requests library.

A `Session` object lets you:

- persist cookies between requests
- reuse TCP connections (faster requests)
- share headers/auth across requests
- manage login states
- configure retries/adapters
- simulate browser-like behavior

If you understand `Session`, you'll feel much more confident building:

- API clients
- web scrapers
- authentication flows
- automation tools
- backend integrations

---

### Basic Difference: `requests.get()` vs Session

Without session:

In [2]:
import requests

response = requests.get("https://httpbin.org/get")
print(response.status_code)

200


Each request is independent.

---

With session:

In [3]:
import requests

session = requests.Session()

response = session.get("https://httpbin.org/get")
print(response.status_code)

200


Now the **session remembers**:

- cookies
- headers
- auth
- connection pool

---

**Why Sessions Matter**

Imagine logging into a website.

**Without a session**:
```py
requests.post(login_url)
requests.get(profile_url)
```
The second request **may not know** you're logged in.

**With a session**:
```py
session = requests.Session()

session.post(login_url)
session.get(profile_url)
```
Cookies **persist automatically**.

**This is how browsers work**.

---

### 1. Persistent Headers

Very common in APIs.

Instead of repeating headers:
```py
headers = {
    "Authorization": "Bearer TOKEN"
}

requests.get(url1, headers=headers)
requests.get(url2, headers=headers)
requests.get(url3, headers=headers)
```
Use a session:

In [4]:
import requests

session = requests.Session()

session.headers.update(
    {
        "Authorization": "Bearer MY_SECRET_TOKEN",
        "User-Agent": "MyPythonApp/1.0"
    }
)

response = session.get("https://httpbin.org/headers")
response.raise_for_status()

In [7]:
response.status_code

200

In [8]:
response.json()

{'headers': {'Accept': '*/*',
  'Accept-Encoding': 'gzip, deflate',
  'Authorization': 'Bearer MY_SECRET_TOKEN',
  'Host': 'httpbin.org',
  'User-Agent': 'MyPythonApp/1.0',
  'X-Amzn-Trace-Id': 'Root=1-6a1952e9-15e82e500f47f62b52d82a66'}}

**What happens?**

**Every request** now automatically includes:
```http
Authorization: Bearer MY_SECRET_TOKEN
User-Agent: MyPythonApp/1.0
```
This is extremely common in:

REST APIs
scraping
automation
authenticated services

---

### 2. Persistent Cookies

One of the MOST important Session features.

**Setting a cookie**

In [ ]:
import requests

session = requests.Session()
# This endpoint sets a cookie.
response = session.get(
    "https://httpbin.org/cookies/set/theme/dark"
)

response.status_code

200

**Reading stored cookies**

In [10]:
response = session.get(
    "https://httpbin.org/cookies"
)

response.json()

{'cookies': {'theme': 'dark'}}

The session remembered the cookie automatically.

**Inspect Session Cookies**

In [7]:
import requests

session = requests.Session()

session.get("https://httpbin.org/cookies/set/user/john")

print(session.cookies)

<RequestsCookieJar[<Cookie user=john for httpbin.org/>]>


**Convert Cookies to Dict**

Useful for debugging.

In [8]:
print(session.cookies.get_dict())

{'user': 'john'}


### 3. Default Query Parameters

You can share parameters across requests.

In [9]:
import requests

session = requests.Session()

session.params = {
    "api_key": "123456"
}

response = session.get(
    "https://httpbin.org/get"
)

print(response.json()["args"])

{'api_key': '123456'}


### 4. Authentication with Session

Very common in APIs.

**Basic Auth Example**

In [10]:
import requests

session = requests.Session()

session.auth = ("user", "passwd")

response = session.get(
    "https://httpbin.org/basic-auth/user/passwd"
)

print(response.status_code)
print(response.json())

200
{'authenticated': True, 'user': 'user'}


### 5. Reusing Connections (Performance)

This is one of the hidden superpowers.

Without Session:

In [12]:
for _ in range(10):
    requests.get("https://httpbin.org/get")

Each request may create a **new TCP connection**.

With Session:

In [ ]:
session = requests.Session()

for _ in range(100):
    session.get("https://httpbin.org/get")

The **connection pool is reused**.

Benefits:

- faster
- lower latency
- fewer TCP handshakes
- better performance

Very important for:

- crawlers
- bots
- API-heavy apps

---

### 6. Sending POST Requests

Sessions work with all HTTP methods.

In [13]:
import requests

session = requests.Session()

response = session.post(
    "https://httpbin.org/post",
    json={
        "username": "john",
        "role": "admin"
    }
)

print(response.json()["json"])

{'role': 'admin', 'username': 'john'}


### 7. Session + JSON APIs

Real-world pattern:

In [14]:
import requests

session = requests.Session()

session.headers.update({
    "Authorization": "Bearer SECRET_TOKEN",
    "Accept": "application/json"
})

response = session.get(
    "https://jsonplaceholder.typicode.com/posts/1"
)

data = response.json()

print(data)

{'userId': 1, 'id': 1, 'title': 'sunt aut facere repellat provident occaecati excepturi optio reprehenderit', 'body': 'quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto'}


### 8. Timeouts (IMPORTANT)

Never skip timeouts in production.

Bad:
```py
requests.get(url)
```

**Could hang forever**.

Good:

In [15]:
response = session.get(
    "https://httpbin.org/delay/3",
    timeout=5
)

print(response.status_code)

200


**Trigger a Timeout Exception**

In [16]:
import requests

session = requests.Session()

try:
    session.get(
        "https://httpbin.org/delay/5",
        timeout=1
    )

except requests.exceptions.Timeout:
    print("Request timed out!")

Request timed out!


### 9. Handling HTTP Errors

**Example: 500 Error**

In [17]:
import requests

response = requests.get(
    "https://httpbin.org/status/500"
)

print(response.status_code)

500


**Using `raise_for_status()`**

In [18]:
import requests

response = requests.get(
    "https://httpbin.org/status/500"
)

try:
    response.raise_for_status()

except requests.exceptions.HTTPError as e:
    print("HTTP Error:", e)

HTTP Error: 500 Server Error: INTERNAL SERVER ERROR for url: https://httpbin.org/status/500


This is a BEST PRACTICE.

---

### 10. Session Context Manager

Very clean approach.

**Automatically closes connections**.

In [19]:
import requests

with requests.Session() as session:

    response = session.get(
        "https://httpbin.org/get"
    )

    print(response.status_code)

200


Equivalent to:
```py
session.close()
```
after the block ends.

---

### 11. Custom Session Headers per Request

Session defaults can still be overridden.

In [20]:
import requests

session = requests.Session()

session.headers.update({
    "User-Agent": "MyApp/1.0"
})

response = session.get(
    "https://httpbin.org/headers",
    headers={
        "User-Agent": "SpecialAgent/9.9"
    }
)

print(response.json())

{'headers': {'Accept': '*/*', 'Accept-Encoding': 'gzip, deflate', 'Host': 'httpbin.org', 'User-Agent': 'SpecialAgent/9.9', 'X-Amzn-Trace-Id': 'Root=1-6a072f98-5e59c61f5a45677a291bd453'}}


Per-request headers override session headers.

---

### 12. Retry Failed Requests (VERY IMPORTANT)

Production-grade feature.

Requests itself does **NOT retry** automatically.

You configure retries using `urllib3`.

**Retry Example**

In [21]:
import requests

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

session = requests.Session()

retry_strategy = Retry(
    total=3,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
)

adapter = HTTPAdapter(
    max_retries=retry_strategy
)

session.mount("https://", adapter)
session.mount("http://", adapter)

response = session.get(
    "https://httpbin.org/status/500"
)

print(response.status_code)

RetryError: HTTPSConnectionPool(host='httpbin.org', port=443): Max retries exceeded with url: /status/500 (Caused by ResponseError('too many 500 error responses'))

**What Happens?**

If server returns:

- 500
- 502
- 503
- 504
- 429

Requests will retry automatically.

This is EXTREMELY useful in real systems.

---

### 13. Connection Adapters

Sessions can mount adapters.

Most common usage:

- retries
- proxies
- custom SSL
- connection pools

Example:

In [ ]:
session.mount("https://", adapter)

Meaning:

> use this adapter for all HTTPS requests

---

### 14. Sending Files

Very common feature.

In [ ]:
import requests

session = requests.Session()

with open("example.txt", "rb") as file:

    response = session.post(
        "https://httpbin.org/post",
        files={
            "file": file
        }
    )

print(response.status_code)

### 15. Inspect Request/Response

Useful for debugging.

In [22]:
import requests

session = requests.Session()

response = session.get(
    "https://httpbin.org/get"
)

print(response.request.headers)
print(response.headers)

{'User-Agent': 'python-requests/2.33.1', 'Accept-Encoding': 'gzip, deflate', 'Accept': '*/*', 'Connection': 'keep-alive'}
{'Date': 'Fri, 15 May 2026 14:42:20 GMT', 'Content-Type': 'application/json', 'Content-Length': '307', 'Connection': 'keep-alive', 'Server': 'gunicorn/19.9.0', 'Access-Control-Allow-Origin': '*', 'Access-Control-Allow-Credentials': 'true'}


### 16. Redirect Handling

Requests follows redirects automatically.

In [23]:
import requests

response = requests.get(
    "http://httpbin.org/redirect/1"
)

print(response.url)

http://httpbin.org/get


Disable redirects:

In [24]:
response = requests.get(
    "http://httpbin.org/redirect/1",
    allow_redirects=False
)

print(response.status_code)
print(response.headers["Location"])

302
/get


### 17. Streaming Responses

Useful for:

- downloading large files
- video streams
- big JSON data

In [25]:
import requests

session = requests.Session()

response = session.get(
    "https://httpbin.org/stream/5",
    stream=True
)

for line in response.iter_lines():

    if line:
        print(line.decode())

{"url": "https://httpbin.org/stream/5", "args": {}, "headers": {"Host": "httpbin.org", "X-Amzn-Trace-Id": "Root=1-6a073147-30cb57401d19057b6cfcac6e", "User-Agent": "python-requests/2.33.1", "Accept-Encoding": "gzip, deflate", "Accept": "*/*"}, "origin": "37.188.207.95", "id": 0}
{"url": "https://httpbin.org/stream/5", "args": {}, "headers": {"Host": "httpbin.org", "X-Amzn-Trace-Id": "Root=1-6a073147-30cb57401d19057b6cfcac6e", "User-Agent": "python-requests/2.33.1", "Accept-Encoding": "gzip, deflate", "Accept": "*/*"}, "origin": "37.188.207.95", "id": 1}
{"url": "https://httpbin.org/stream/5", "args": {}, "headers": {"Host": "httpbin.org", "X-Amzn-Trace-Id": "Root=1-6a073147-30cb57401d19057b6cfcac6e", "User-Agent": "python-requests/2.33.1", "Accept-Encoding": "gzip, deflate", "Accept": "*/*"}, "origin": "37.188.207.95", "id": 2}
{"url": "https://httpbin.org/stream/5", "args": {}, "headers": {"Host": "httpbin.org", "X-Amzn-Trace-Id": "Root=1-6a073147-30cb57401d19057b6cfcac6e", "User-Agen